In [3]:
# Libraries import

import requests
import os
import pygsheets
import pandas as pd
from dotenv import load_dotenv
from pangres import upsert
from sqlalchemy import text, create_engine

# Load environment variables
load_dotenv()

API_KEY = os.getenv('riot_api_key')
username=os.getenv('db_username')
password=os.getenv('db_password')
host=os.getenv('db_host')
port=os.getenv('db_port')
name=os.getenv('db_name')

# Create a database connection string

def create_db_connection_string(username, password, host, port, name):
    connection_url = 'postgresql+psycopg2://'+username+':'+password+'@'+host+':'+port+'/'+name
    return connection_url

conn = create_db_connection_string(username, password, host, port, name)

db_engine = create_engine(conn, pool_recycle=3600)

connection = db_engine.connect()

In [5]:
# Get Challengers summonerId

def get_ladder(top=None):
    root = 'https://br1.api.riotgames.com/tft/'
    challenger = 'league/v1/challenger?queue=RANKED_TFT'
    grandmaster = 'league/v1/grandmaster?queue=RANKED_TFT'
    master = 'league/v1/master?queue=RANKED_TFT'

    challenger_response = requests.get(root + challenger + '&api_key=' + API_KEY)
    
    challenger_df = pd.DataFrame(challenger_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    grandmaster_df = pd.DataFrame()
    master_df = pd.DataFrame()
    
    if top > 50:
        grandmaster_response = requests.get(root + grandmaster + '&api_key=' + API_KEY)
        grandmaster_df = pd.DataFrame(grandmaster_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    
    if top > 150:
        master_response = requests.get(root + master + '&api_key=' + API_KEY)
        master_df = pd.DataFrame(master_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    ladder = pd.concat([challenger_df, grandmaster_df, master_df])[:top].reset_index(drop=True)
    ladder = ladder.reset_index(drop=False).drop(columns='rank').rename(columns={'index':'rank'})
    ladder['rank'] += 1

    return ladder

In [7]:
df_ladder = get_ladder(top=250)
df_ladder = df_ladder.set_index('rank')
df_ladder

,summonerId,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
rank,,,,,,,,
1,81TNb5AeP0llAXhrPPctqpn29ySkvIF9j_PJHLEOvMDXAeo,1789,224,97,True,False,False,False
2,h7ZPEa9uBc7YUhKgYYqSrVFK-Kt-lLQeDBKDP66NVROn-w,1732,158,57,False,False,False,True
3,8ZNCE5oDTwbMwVBmhOZzhOxXJampyL-SWWvA1ZkidSFytQ,1651,269,131,True,False,False,False
4,FCA6EjmiXnKVZd3YGVykmsSJCkS9MZ3a6ecZCWsso6JHtLY,1633,176,78,True,False,False,True
5,HHzxnU-gmYi4jX3TPyB_T-Ktp_y2n4ByVHUcrzDZTF4dx_0,1632,496,299,True,False,False,False
...,...,...,...,...,...,...,...,...
246,RC9PJx2_FBuPY16iqaMlOAbDo0iWQ-DyuyO94u7jgxevmA,340,160,141,True,False,False,False
247,1nQUl8yo3SQz-Qs-FhMRaspj9VpYdB3wZlIPWC3ZfhNKbQ,340,140,94,False,False,False,False
248,0dFnH9ZMoDPnhEAIUqWSimqtnB0yPypB6nTK1gY3uc_pwCo,338,198,188,True,False,False,False


In [9]:
upsert(con=connection, df=df_ladder, schema='tft_ranked', table_name='ranked_ladder', create_table=True, create_schema=True, if_row_exists='update')

In [10]:
connection.commit()

In [11]:
with db_engine.connect() as conn:
    df_query = pd.read_sql(text("SELECT * FROM tft_ranked.ranked_ladder"), conn)

In [12]:
df_query

,rank,summonerId,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
0,1,81TNb5AeP0llAXhrPPctqpn29ySkvIF9j_PJHLEOvMDXAeo,1789,224,97,True,False,False,False
1,2,h7ZPEa9uBc7YUhKgYYqSrVFK-Kt-lLQeDBKDP66NVROn-w,1732,158,57,False,False,False,True
2,3,8ZNCE5oDTwbMwVBmhOZzhOxXJampyL-SWWvA1ZkidSFytQ,1651,269,131,True,False,False,False
3,4,FCA6EjmiXnKVZd3YGVykmsSJCkS9MZ3a6ecZCWsso6JHtLY,1633,176,78,True,False,False,True
4,5,HHzxnU-gmYi4jX3TPyB_T-Ktp_y2n4ByVHUcrzDZTF4dx_0,1632,496,299,True,False,False,False
...,...,...,...,...,...,...,...,...,...
245,246,RC9PJx2_FBuPY16iqaMlOAbDo0iWQ-DyuyO94u7jgxevmA,340,160,141,True,False,False,False
246,247,1nQUl8yo3SQz-Qs-FhMRaspj9VpYdB3wZlIPWC3ZfhNKbQ,340,140,94,False,False,False,False
247,248,0dFnH9ZMoDPnhEAIUqWSimqtnB0yPypB6nTK1gY3uc_pwCo,338,198,188,True,False,False,False
248,249,YSvDnL_5R8HAyuehkplKkvjqHFZb0OgW8gGld1TDDnIvoOQ,336,93,48,False,False,False,True


In [ ]:
game_name = 'LustGuard'
tag_line = 'BR1'

def get_puuid(game_name=None, tag_line=None, API_KEY=None):
    link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}?api_key={API_KEY}'
    response = requests.get(link)
    return response.json()['puuid']

In [ ]:
get_puuid(game_name=game_name, tag_line=tag_line, API_KEY=API_KEY)

In [ ]:
temp_df = get_ladder(top=100)['summonerId']

In [ ]:
puuid_dict = {}
def get_puuid(df):
    root = 'https://br1.api.riotgames.com/tft/league/v1/entries/by-summoner/'
    for summoner in temp_df:
        response = requests.get(root + summoner + '?api_key=' + API_KEY)
        if response.status_code == 200:
            data = response.json()
            summoner_puuid = data[0].get('puuid')
            puuid_dict[summoner] = summoner_puuid
        else:
            print(f'Error to retrieve the data from {summoner}: {response.status_code}')
    return puuid_dict

In [ ]:
df = get_puuid(temp_df)
df